# Connector: notifiers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_notifier.ipynb)

Notifiers push events (`trained`, `shadow`, `ready_for_approval`, `live`, `drift`) to chat. All HTTP here is **mocked** with `respx`, so we can show exactly what each notifier would send.

In [1]:
%pip install -q pecca respx

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
import httpx, respx
from pecca import connectors
from pecca.templates_util import render

payload = {"call": "prod/autoresponse/route_rfi", "event": "ready_for_approval", "version": "v3", "message": "ticket MRM-7"}
text = render("slack_event.md.j2", **payload)
print(text)

*Pecca* `prod/autoresponse/route_rfi` — ready_for_approval
version v3 ticket MRM-7



In [4]:
sent = {}
with respx.mock:
    for name, cfg in {"webhook": {"type": "webhook", "url": "https://hooks.example/generic"},
                      "slack": {"type": "slack", "webhook": "https://hooks.example/slack", "channel": "#ai-platform"},
                      "teams": {"type": "teams", "webhook": "https://hooks.example/teams"}}.items():
        route = respx.post(cfg.get("url") or cfg["webhook"]).mock(return_value=httpx.Response(200))
        connectors.create("notifier", cfg).send(payload["event"], payload, text)
        sent[name] = json.loads(route.calls[0].request.content)
for name, body in sent.items():
    print(f"--- {name}")
    print(json.dumps(body, indent=2)[:700])

--- webhook
{
  "event": "ready_for_approval",
  "payload": {
    "call": "prod/autoresponse/route_rfi",
    "event": "ready_for_approval",
    "version": "v3",
    "message": "ticket MRM-7"
  },
  "text": "*Pecca* `prod/autoresponse/route_rfi` \u2014 ready_for_approval\nversion v3 ticket MRM-7\n"
}
--- slack
{
  "text": "*Pecca* `prod/autoresponse/route_rfi` \u2014 ready_for_approval\nversion v3 ticket MRM-7\n",
  "channel": "#ai-platform"
}
--- teams
{
  "type": "message",
  "attachments": [
    {
      "contentType": "application/vnd.microsoft.card.adaptive",
      "contentUrl": null,
      "content": {
        "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
        "type": "AdaptiveCard",
        "version": "1.4",
        "body": [
          {
            "type": "TextBlock",
            "text": "Pecca: ready_for_approval",
            "weight": "Bolder"
          },
          {
            "type": "TextBlock",
            "text": "*Pecca* `prod/autoresponse/route_

In a project, configure one under `integrations.notifier`; `events:` filters which events are sent, and a failing notifier never affects the pipeline.

In [5]:
Path("pecca.yaml").write_text("""version: 1
projects:
  default:
    governance: {extends: none}
    integrations:
      notifier: {type: slack, webhook: "https://hooks.example/slack", channel: "#ai-platform", events: [trained, shadow, live]}
""")
context.reset()
with respx.mock:
    route = respx.post("https://hooks.example/slack").mock(return_value=httpx.Response(200))
    train_demo()
    pecca.promote("route_rfi", "shadow")
print([json.loads(c.request.content)["text"].splitlines()[0] for c in route.calls])

['*Pecca* `default/default/route_rfi` — trained', '*Pecca* `default/default/route_rfi` — shadow']
